# Exploração inicial — Preços de combustíveis (ANP)

Objetivo: entender a estrutura do arquivo antes de montar o pipeline de carga.
Arquivo usado: `Preços semestrais - AUTOMOTIVOS_2025.01.csv` (1º semestre de 2025).
Como todos os semestres seguem o mesmo formato, o que for descoberto aqui vale para os outros arquivos.

In [26]:
import pandas as pd

caminho = "../dados/raw/Preços semestrais - AUTOMOTIVOS_2025.01.csv"

## 1. Leitura do arquivo

A primeira tentativa com `pd.read_csv` padrão deu `ParserError`.
Para entender o motivo, li as primeiras linhas como texto puro.

In [27]:
with open(caminho, encoding="utf-8", errors="replace") as arquivo:
    for _ in range(3) :
        print(arquivo.readline())

Regiao - Sigla;Estado - Sigla;Municipio;Revenda;CNPJ da Revenda;Nome da Rua;Numero Rua;Complemento;Bairro;Cep;Produto;Data da Coleta;Valor de Venda;Valor de Compra;Unidade de Medida;Bandeira

NE;CE;SOBRAL;ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. E GÁS NAT; 08.775.979/0002-62;RUA TABELIÃO IDELFONSO CAVALCANTI;455;;CENTRO;62010-000;GASOLINA;01/01/2025;6,29;;R$ / litro;RAIZEN

NE;CE;SOBRAL;ECONOGÁS DO BRASIL DIST. DERIV. DE PET. BIOC. E GÁS NAT; 08.775.979/0002-62;RUA TABELIÃO IDELFONSO CAVALCANTI;455;;CENTRO;62010-000;GASOLINA ADITIVADA;01/01/2025;6,49;;R$ / litro;RAIZEN



**Descobertas:**
- O separador é `;`, não vírgula.
- O encoding é UTF-8 (os acentos aparecem corretos).
- Os preços usam vírgula como separador decimal (padrão brasileiro).
- Alguns campos vêm com espaço sobrando no início (ex.: CNPJ).

In [28]:
df = pd.read_csv(caminho, sep=";", encoding="utf-8", decimal=",", low_memory=False)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 429523 entries, 0 to 429522
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Regiao - Sigla     420409 non-null  str    
 1   Estado - Sigla     420409 non-null  str    
 2   Municipio          420409 non-null  str    
 3   Revenda            420409 non-null  str    
 4   CNPJ da Revenda    420409 non-null  str    
 5   Nome da Rua        420409 non-null  str    
 6   Numero Rua         420351 non-null  str    
 7   Complemento        96615 non-null   str    
 8   Bairro             419547 non-null  str    
 9   Cep                420409 non-null  str    
 10  Produto            420409 non-null  str    
 11  Data da Coleta     420409 non-null  str    
 12  Valor de Venda     420409 non-null  float64
 13  Valor de Compra    0 non-null       float64
 14  Unidade de Medida  420409 non-null  str    
 15  Bandeira           420409 non-null  str    
dtypes: float64(2)

## 2. Visão geral das colunas

- `Valor de Compra` está 100% vazia → será descartada.
- `Data da Coleta` está como texto → precisa ser convertida para data.
- `Complemento` está quase toda vazia, o que é normal para endereço.
- O arquivo tem 429.523 linhas, mas as colunas têm só 420.409 valores preenchidos.
- Com `decimal=","`, o `Valor de Venda` passou a ser lido como número (float).

In [29]:
vazias = df[df["Regiao - Sigla"].isna()]
print(len(vazias))
vazias.head()

9114


,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira
420409,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
420410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
420411,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
420412,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
420413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# how="all": remove só linhas 100% vazias (sem isso, perderia linhas com Complemento vazio)
df = df.dropna(how="all")
print(len(df))

420409


## 3. Linhas vazias

Havia 9.114 linhas totalmente em branco, todas no final do arquivo (sobra da exportação).
Removidas com `dropna(how="all")`. Resultado: 420.409 linhas válidas.

## 4. Conteúdo das Colunas

In [31]:
df["Produto"].value_counts()

Produto
GASOLINA              110818
ETANOL                 92598
GASOLINA ADITIVADA     85580
DIESEL S10             76134
DIESEL                 45771
GNV                     9508
Name: count, dtype: int64

In [32]:
df["Unidade de Medida"].value_counts()

Unidade de Medida
R$ / litro    410901
R$ / m³         9508
Name: count, dtype: int64

**Produtos e unidades:**
- São 6 produtos: gasolina, gasolina aditivada, etanol, diesel, diesel S10 e GNV.
- O GNV é vendido em R$/m³ e os demais em R$/litro (as 9.508 coletas de GNV batem com as 9.508 em m³).
- Por isso, o preço do GNV não pode ser comparado nem somado com o dos outros combustíveis.

In [35]:
df["Bandeira"].nunique()

46

In [36]:
df["Bandeira"].value_counts().head(10)

Bandeira
BRANCA      140728
VIBRA        92844
IPIRANGA     88305
RAIZEN       62023
ALE          11664
SABBÁ         4450
RODOIL        1877
ATEM' S       1836
CHARRUA       1786
SP            1313
Name: count, dtype: int64

**Bandeiras:**
- Existem 46 bandeiras diferentes.
- A maior é "BRANCA" (cerca de 1/3 das coletas), que representa postos sem bandeira, os independentes.
- Depois vêm Vibra (antiga BR), Ipiranga e Raízen.
- Pergunta para a análise: posto bandeira branca é mais barato que posto de marca?

In [37]:
df["Valor de Venda"].describe()

count    420409.000000
mean          5.890884
std           0.893260
min           3.190000
25%           5.590000
50%           6.170000
75%           6.490000
max           9.490000
Name: Valor de Venda, dtype: float64

**Preço geral:**
O `describe()` geral mistura todos os combustíveis (inclusive o GNV, em outra unidade), então a média de R$ 5,89 não representa nenhum produto. É preciso analisar por produto.

In [38]:
df["Data da Coleta"].min(), df["Data da Coleta"].max()

('01/01/2025', '31/03/2025')

**Atenção:** o resultado indica coletas só até 31/03, mas o arquivo é do semestre inteiro.
Isso acontece porque a data ainda está como texto, e o Python compara texto caractere por caractere, começando pelo dia. Por isso "31/03" aparece como maior que "30/06".
Solução: converter a coluna para o tipo data.

In [39]:
df["Data da Coleta"] = pd.to_datetime(df["Data da Coleta"], format="%d/%m/%Y")
df["Data da Coleta"].min(), df["Data da Coleta"].max()  

(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-06-30 00:00:00'))

Depois da conversão, o período real vai de 01/01 até 30/06 de 2025.

In [40]:
df.groupby("Produto")["Valor de Venda"].describe()

,count,mean,std,min,25%,50%,75%,max
Produto,,,,,,,,
DIESEL,45771.0,6.239395,0.399908,5.29,5.97,6.19,6.45,8.33
DIESEL S10,76134.0,6.291569,0.413169,5.19,5.99,6.29,6.49,8.99
ETANOL,92598.0,4.477963,0.458766,3.19,4.14,4.47,4.79,6.89
GASOLINA,110818.0,6.288020,0.405233,4.99,5.99,6.28,6.49,8.99
GASOLINA ADITIVADA,85580.0,6.482922,0.409874,5.19,6.21,6.49,6.69,9.49
GNV,9508.0,4.807552,0.434664,3.79,4.49,4.74,4.99,6.49


**Preço por produto (1º semestre de 2025):**
- Gasolina comum: média de R$ 6,29. A aditivada sai cerca de R$ 0,19 mais cara (R$ 6,48).
- Diesel S10 (R$ 6,29) custa praticamente o mesmo que a gasolina; o diesel comum fica em R$ 6,24.
- Etanol: média de R$ 4,48, o que equivale a 71% do preço da gasolina, logo acima da regra dos 70%. Na média nacional, o etanol fica no limite de compensar. Vale analisar por estado.
- GNV: R$ 4,81 por m³ (unidade diferente, não comparável).
- A dispersão é baixa (desvio padrão em torno de R$ 0,40), mas os máximos chegam a R$ 8,99 (gasolina) e R$ 9,49 (aditivada). Investigar quais regiões têm esses preços.

In [41]:
df.duplicated().sum()

np.int64(0)

In [42]:
df["CNPJ da Revenda"].str.startswith(" ").sum()

np.int64(420409)

In [43]:
df["CNPJ da Revenda"] = df["CNPJ da Revenda"].str.strip()
df["CNPJ da Revenda"].str.startswith(" ").sum()

np.int64(0)

## 5. Resumo: o que o pipeline precisa tratar

**Leitura**
- `sep=";"`, `encoding="utf-8"`, `decimal=","`, `low_memory=False`

**Limpeza**
- Remover linhas 100% vazias no final do arquivo (`dropna(how="all")`)
- Remover a coluna `Valor de Compra` (100% vazia)
- Converter `Data da Coleta` para data (`format="%d/%m/%Y"`)
- Remover espaços sobrando nos textos: todos os 420.409 CNPJs vinham com um espaço no início (padrão da exportação da ANP). Testado com `.str.strip()`, que resolveu.
- Duplicatas: nenhuma encontrada (manter a verificação no pipeline, caso apareçam em outros semestres)

**Cuidados na análise**
- GNV é medido em R$/m³; os demais em R$/litro. Não misturar nas médias.
- "BRANCA" na coluna Bandeira = posto sem bandeira (independente).
- Comparar preços sempre por produto.